#**TRANSFORM MODEL**



## **INSTALL AND IMPORT LIBRARIES**



In [ ]:
!pip install yfinance plotly scikit-learn torch -q

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yfinance as yf
import plotly.graph_objects as go
import plotly.subplots as sp
import os
import copy
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)
from torch.utils.data import DataLoader, TensorDataset

##**CONFIG**

In [2]:
INPUT_LEN  = 90
OUTPUT_LEN = 5
EPOCHS     = 25
BATCH_SIZE = 64
TICKERS = [
    "RELIANCE.NS","TCS.NS","HDFCBANK.NS","ICICIBANK.NS","KOTAKBANK.NS",
    "SBIN.NS","AXISBANK.NS","BAJFINANCE.NS","BAJAJFINSV.NS","HDFCLIFE.NS",
    "SBILIFE.NS","ITC.NS","HINDUNILVR.NS","NESTLEIND.NS","TATACONSUM.NS",
    "BRITANNIA.NS","ASIANPAINT.NS","ULTRACEMCO.NS","GRASIM.NS","LT.NS",
    "ADANIENT.NS","ADANIPORTS.NS","BHARTIARTL.NS","INDUSINDBK.NS","MARUTI.NS",
    "HEROMOTOCO.NS","EICHERMOT.NS","BAJAJ-AUTO.NS","POWERGRID.NS","NTPC.NS",
    "ONGC.NS","COALINDIA.NS","JSWSTEEL.NS","TATASTEEL.NS","HCLTECH.NS",
    "WIPRO.NS","TECHM.NS","INFY.NS","CIPLA.NS","DRREDDY.NS",
    "APOLLOHOSP.NS","SUNPHARMA.NS","DIVISLAB.NS","LTIM.NS","TITAN.NS",
    "INDIGO.NS","TRENT.NS","JIOFIN.NS"
]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## **DATA PIPELINE**



In [3]:
ticker_to_id = {t: i for i, t in enumerate(TICKERS)}

class DataPipeline:
    def __init__(self):
        self.scalers  = {}
        self.date_map = {}

    def fetch_data(self, ticker):
        df = yf.download(ticker, period="15y", progress=False)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df.rename(columns=str.lower)
        df = df[["open", "high", "low", "close", "volume"]]
        return df.dropna()

    def add_features(self, df):
        df = df.copy()
        df["return_1d"]   = df["close"].pct_change()
        df["return_5d"]   = df["close"].pct_change(5)
        df["sma_20"]      = df["close"].rolling(20).mean()
        df["ema_12"]      = df["close"].ewm(span=12).mean()
        df["ema_26"]      = df["close"].ewm(span=26).mean()
        df["macd"]        = df["ema_12"] - df["ema_26"]
        delta = df["close"].diff()
        gain  = delta.clip(lower=0).rolling(14).mean()
        loss  = (-delta.clip(upper=0)).rolling(14).mean()
        rs    = gain / loss.replace(0, np.nan)
        df["rsi"]         = 100 - (100 / (1 + rs))
        std = df["close"].rolling(20).std()
        df["bb_width"]    = (2 * std) / df["sma_20"]
        df["volatility"]  = df["return_1d"].rolling(20).std()
        df["volume_ratio"]= df["volume"] / df["volume"].rolling(20).mean()
        df = df.replace([np.inf, -np.inf], np.nan)
        return df.dropna()

    def create_sequences(self, data, sid):
        X, y, s = [], [], []
        for i in range(len(data) - INPUT_LEN - OUTPUT_LEN):
            X.append(data[i:i + INPUT_LEN])
            y.append(data[i + INPUT_LEN:i + INPUT_LEN + OUTPUT_LEN, 3])
            s.append(sid)
        return np.array(X), np.array(y), np.array(s)

    def prepare(self):
        X_tr, y_tr, s_tr = [], [], []
        X_te, y_te, s_te = [], [], []
        for ticker in TICKERS:
            df  = self.fetch_data(ticker)
            df  = self.add_features(df)
            self.date_map[ticker] = df.index
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(df.values)
            self.scalers[ticker] = scaler
            X, y, s = self.create_sequences(scaled, ticker_to_id[ticker])
            split = int(len(X) * 0.7)
            X_tr.extend(X[:split]); y_tr.extend(y[:split]); s_tr.extend(s[:split])
            X_te.extend(X[split:]); y_te.extend(y[split:]); s_te.extend(s[split:])
        return {
            "train": (np.array(X_tr), np.array(y_tr), np.array(s_tr)),
            "test":  (np.array(X_te), np.array(y_te), np.array(s_te))
        }

    def prepare_single(self, ticker, fit_scaler=True, scaler=None):
        df = self.fetch_data(ticker)
        df = self.add_features(df)
        self.date_map[ticker] = df.index
        if fit_scaler:
            scaler = MinMaxScaler()
            scaler.fit(df.values)
        self.scalers[ticker] = scaler
        scaled = scaler.transform(df.values)

        X, y, s = self.create_sequences(scaled, sid=0)
        split   = int(len(X) * 0.7)
        return (
            X[:split],  y[:split],
            X[split:],  y[split:],
            df.index
        )

## **MODEL**

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        pe = torch.zeros(500, d_model)
        pos = torch.arange(0, 500).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(x.device)


class Model(nn.Module):
    def __init__(self, input_size, num_stocks):
        super().__init__()
        self.use_embedding = num_stocks > 1
        self.proj  = nn.Linear(input_size, 64)
        self.pos   = PositionalEncoding(64)
        if self.use_embedding:
            self.embed = nn.Embedding(num_stocks, 64)
        enc = nn.TransformerEncoderLayer(d_model=64, nhead=4, batch_first=True)
        self.trans = nn.TransformerEncoder(enc, num_layers=2)
        self.fc    = nn.Linear(64, OUTPUT_LEN)

    def forward(self, x, sid=None):
        x = self.proj(x)
        x = self.pos(x)
        if self.use_embedding and sid is not None:
            x = x + self.embed(sid).unsqueeze(1)
        x = self.trans(x)
        return self.fc(x[:, -1, :])

## **TRAIN MODEL**

In [22]:
pipeline = DataPipeline()
MODEL_PATH = "nifty50_model.pt"

if os.path.exists(MODEL_PATH):
    print(f"Loading model from {MODEL_PATH} ...")
    checkpoint = torch.load(MODEL_PATH, map_location=device, weights_only=False)
    INPUT_FEATS  = checkpoint["input_size"]
    model        = Model(INPUT_FEATS, len(TICKERS)).to(device)
    model.load_state_dict(checkpoint["model_state"])
    print(f"Model loaded (input_size={INPUT_FEATS})")
    if "scalers" in checkpoint:
        pipeline.scalers  = checkpoint["scalers"]
        pipeline.date_map = checkpoint["date_map"]
        print(" Scalers loaded from checkpoint")
    else:
        print("  No scalers in checkpoint — will re-fetch training data for scalers.")
        print("    (This takes a few minutes but only runs once.)")
        splits = pipeline.prepare()
        X_train = torch.tensor(splits["train"][0], dtype=torch.float32)
        INPUT_FEATS = X_train.shape[2]
else:
    print(" No checkpoint found — training from scratch (this will take a while).")
    print("    Upload your .pt file to skip this step next time.")
    splits  = pipeline.prepare()
    X_train, y_train, s_train = splits["train"]
    INPUT_FEATS = X_train.shape[2]

    X_tr = torch.tensor(X_train, dtype=torch.float32)
    y_tr = torch.tensor(y_train, dtype=torch.float32)
    s_tr = torch.tensor(s_train, dtype=torch.long)
    loader = DataLoader(TensorDataset(X_tr, y_tr, s_tr),
                        batch_size=BATCH_SIZE, shuffle=True)

    model     = Model(INPUT_FEATS, len(TICKERS)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
    loss_fn   = nn.MSELoss()

    for epoch in range(EPOCHS):
        model.train()
        total = 0
        for xb, yb, sb in loader:
            xb, yb, sb = xb.to(device), yb.to(device), sb.to(device)
            pred = model(xb, sb)
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total += loss.item()
        avg = total / len(loader)
        scheduler.step(avg)
        print(f"Epoch {epoch+1}/{EPOCHS}: loss={avg:.4f}")

    # Save for next time
    torch.save({
        "model_state": model.state_dict(),
        "input_size":  INPUT_FEATS,
        "scalers":     pipeline.scalers,
        "date_map":    pipeline.date_map,
    }, MODEL_PATH)
    print(f" Model saved to {MODEL_PATH}")

model.eval()
print("Model ready")

Loading model from nifty50_model.pt ...
Model loaded (input_size=15)
 Scalers loaded from checkpoint
Model ready


In [7]:
from google.colab import files
files.download("nifty50_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **FETCH & PREPARE THE UNSEEN STOCK**

In [9]:
UNSEEN_TICKER=input("Stock Name:")
print(f"Fetching {UNSEEN_TICKER} ...")
(
    X_unseen_train, y_unseen_train,
    X_unseen_test,  y_unseen_test,
    unseen_dates
) = pipeline.prepare_single(UNSEEN_TICKER)

print(f"{UNSEEN_TICKER} | Train sequences: {len(X_unseen_train)} | Test sequences: {len(X_unseen_test)}")
print(f"   Features per timestep: {X_unseen_test.shape[2]} (expected {INPUT_FEATS})")

assert X_unseen_test.shape[2] == INPUT_FEATS, \
    f"Feature mismatch! Got {X_unseen_test.shape[2]}, expected {INPUT_FEATS}"

Stock Name:DMART.NS
Fetching DMART.NS ...


/tmp/ipykernel_5655/3284724100.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="15y", progress=False)


DMART.NS | Train sequences: 1480 | Test sequences: 635
   Features per timestep: 15 (expected 15)


## **USING INVERSE TRANSFORM**

In [10]:
def inverse_close(arr, ticker):
    scaler = pipeline.scalers[ticker]
    dummy  = np.zeros((len(arr), INPUT_FEATS))
    dummy[:, 3] = arr          # col 3 = close
    return scaler.inverse_transform(dummy)[:, 3]

def run_inference(model_to_use, X, emb_override=None):
    model_to_use.eval()
    preds = []
    with torch.no_grad():
        for i in range(len(X)):
            x = torch.tensor(X[i], dtype=torch.float32).unsqueeze(0).to(device)
            if emb_override is not None:

                xp = model_to_use.proj(x)
                xp = model_to_use.pos(xp)
                xp = xp + emb_override.unsqueeze(1).to(device)
                xp = model_to_use.trans(xp)
                out = model_to_use.fc(xp[:, -1, :])
            else:
                out = model_to_use(x)
            preds.append(out.cpu().numpy()[0])
    return np.array(preds)

## **USING FEW SHOT**

In [11]:
print("FEW-SHOT ADAPTATION")

fs_model = copy.deepcopy(model)
FEWSHOT_EPOCHS = 8
FEWSHOT_LR     = 0.001
new_emb_weight = torch.zeros(1, 64)

if model.use_embedding:
    mean_emb = model.embed.weight.mean(dim=0, keepdim=True)
    new_emb_weight = mean_emb.cpu().detach().clone()

unseen_emb = nn.Parameter(new_emb_weight.to(device), requires_grad=True)

for param in fs_model.parameters():
    param.requires_grad = False

fs_optimizer = torch.optim.Adam([unseen_emb], lr=FEWSHOT_LR)
fs_loss_fn   = nn.MSELoss()

X_fs = torch.tensor(X_unseen_train, dtype=torch.float32)
y_fs = torch.tensor(y_unseen_train, dtype=torch.float32)
fs_loader = DataLoader(TensorDataset(X_fs, y_fs),
                       batch_size=32, shuffle=True)
fs_model.train()
print(f"\nTraining new embedding on {len(X_fs)} sequences for {FEWSHOT_EPOCHS} epochs ...")

for epoch in range(FEWSHOT_EPOCHS):
    total = 0
    for xb, yb in fs_loader:
        xb, yb = xb.to(device), yb.to(device)

        xp   = fs_model.proj(xb)
        xp   = fs_model.pos(xp)
        xp   = xp + unseen_emb.unsqueeze(1).expand(xb.size(0), -1, -1)
        xp   = fs_model.trans(xp)
        pred = fs_model.fc(xp[:, -1, :])
        loss = fs_loss_fn(pred, yb)
        fs_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fs_model.parameters(), 1.0)
        fs_optimizer.step()
        total += loss.item()
    print(f"  Epoch {epoch+1}/{FEWSHOT_EPOCHS} | loss={total/len(fs_loader):.5f}")

fs_preds_scaled = run_inference(fs_model, X_unseen_test,
                                emb_override=unseen_emb.detach())

fs_actual = inverse_close(y_unseen_test[:, 0], UNSEEN_TICKER)
fs_pred   = inverse_close(fs_preds_scaled[:, 0], UNSEEN_TICKER)

fs_rmse = np.sqrt(mean_squared_error(fs_actual, fs_pred))
fs_mae  = mean_absolute_error(fs_actual, fs_pred)
fs_r2   = r2_score(fs_actual, fs_pred)
fs_mape = np.mean(np.abs((fs_actual - fs_pred) / fs_actual)) * 100

print(f"\nFew-Shot Metrics — {UNSEEN_TICKER}")
print(f"  MAPE  : {fs_mape:.2f}%")
print(f"  RMSE  : ₹{fs_rmse:.2f}")
print(f"  MAE   : ₹{fs_mae:.2f}")
print(f"  R²    : {fs_r2:.4f}")

FEW-SHOT ADAPTATION

Training new embedding on 1480 sequences for 8 epochs ...
  Epoch 1/8 | loss=0.00065
  Epoch 2/8 | loss=0.00061
  Epoch 3/8 | loss=0.00059
  Epoch 4/8 | loss=0.00058
  Epoch 5/8 | loss=0.00057
  Epoch 6/8 | loss=0.00057
  Epoch 7/8 | loss=0.00060
  Epoch 8/8 | loss=0.00055

Few-Shot Metrics — DMART.NS
  MAPE  : 1.95%
  RMSE  : ₹105.79
  MAE   : ₹79.17
  R²    : 0.9473


## **FUTURE FORECAST/PREDICTION**

In [12]:
print(f" Forecasting next {OUTPUT_LEN} trading days for {UNSEEN_TICKER} ...\n")

last_seq = torch.tensor(X_unseen_test[-1], dtype=torch.float32).unsqueeze(0).to(device)

fs_model.eval()
with torch.no_grad():
    xp   = fs_model.proj(last_seq)
    xp   = fs_model.pos(xp)
    xp   = xp + unseen_emb.detach().unsqueeze(1)
    xp   = fs_model.trans(xp)
    fut_scaled = fs_model.fc(xp[:, -1, :]).cpu().numpy()[0]

fut_prices   = inverse_close(fut_scaled, UNSEEN_TICKER)
last_date    = unseen_dates[-1]
future_dates = [last_date + timedelta(days=i+1) for i in range(OUTPUT_LEN)]

print(f"Last known close: ₹{fs_actual[-1]:.2f}  (date: {last_date.date()})")
print()
for d, p in zip(future_dates, fut_prices):
    change = ((p - fs_actual[-1]) / fs_actual[-1]) * 100
    print(f"  {d.date()}  →  ₹{p:.2f}")


N_plot_hist = 30


plot_actual_dates = unseen_dates[-N_plot_hist:]
plot_actual_prices = fs_actual[-N_plot_hist:]
plot_predicted_prices = fs_pred[-N_plot_hist:]


forecast_start_date = unseen_dates[-1]
forecast_start_price = fs_pred[-1]

forecast_dates_combined = [forecast_start_date] + future_dates
forecast_prices_combined = [forecast_start_price] + list(fut_prices)

fig = go.Figure()


fig.add_trace(go.Scatter(
    x=plot_actual_dates, y=plot_actual_prices,
    name="Actual Prices", mode="lines",
    line=dict(color="#00E5FF", width=2)
))

fig.add_trace(go.Scatter(
    x=plot_actual_dates, y=plot_predicted_prices,
    name="Few-Shot Predicted Prices", mode="lines",
    line=dict(color="#69FF47", width=2)
))


fig.add_trace(go.Scatter(
    x=forecast_dates_combined,
    y=forecast_prices_combined,
    name=f"{OUTPUT_LEN}-Day Future Forecast", mode="lines",
    line=dict(color="#FFD700", width=2, dash='dot'),
    marker=dict(size=6, symbol='circle')
))

fig.update_layout(
    title=f"{UNSEEN_TICKER} - Last {N_plot_hist} Days and {OUTPUT_LEN}-Day Forecast",
    xaxis_title="Date",
    yaxis_title="Close Price (₹)",
    template="plotly_dark",
    legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
)

fig.show()

 Forecasting next 5 trading days for DMART.NS ...

Last known close: ₹3770.80  (date: 2026-03-30)

  2026-03-31  →  ₹3820.55
  2026-04-01  →  ₹3839.03
  2026-04-02  →  ₹3844.10
  2026-04-03  →  ₹3836.61
  2026-04-04  →  ₹3837.41


## **MULTIPLE UNSEEN STOCK**

In [17]:
BATCH_UNSEEN = ["PIDILITIND.NS", "HAVELLS.NS", "DABUR.NS"]
BATCH_UNSEEN = [t for t in BATCH_UNSEEN if t not in TICKERS]

results = []

for t in BATCH_UNSEEN:
    try:
        print(f"\n Processing {t} ...", end=" ")
        _, _, X_test_t, y_test_t, dates_for_t = pipeline.prepare_single(t)

        if len(X_test_t) < 10:
            print("  Not enough data, skipping.")
            continue


        p_scaled = run_inference(model, X_test_t, emb_override=mean_emb)
        act = inverse_close(y_test_t[:, 0], t)
        pre = inverse_close(p_scaled[:, 0], t)
        mape = np.mean(np.abs((act - pre) / act)) * 100
        r2   = r2_score(act, pre)
        results.append({"Ticker": t, "MAPE %": round(mape, 2), "R²": round(r2, 4)})
        print(f"MAPE={mape:.2f}%  R²={r2:.4f}")


        print(f"  Forecasting next {OUTPUT_LEN} trading days for {t} ...")
        last_seq_t = X_test_t[-1]


        fut_preds_scaled_t = run_inference(model, np.array([last_seq_t]), emb_override=mean_emb)
        fut_prices_t = inverse_close(fut_preds_scaled_t[0], t)

        last_date_t = dates_for_t[-1]
        future_dates_t = [last_date_t + timedelta(days=i+1) for i in range(OUTPUT_LEN)]

        print(f"  Last known close: ₹{act[-1]:.2f}  (date: {last_date_t.date()})")
        print("  Future Forecast:")
        for d, p in zip(future_dates_t, fut_prices_t):
            print(f"    {d.date()}  →  ₹{p:.2f}")

    except Exception as e:
        print(f" Error: {e}")

if results:
    print("\nBatch Zero-Shot Results")
    print(pd.DataFrame(results).to_string(index=False))
else:
    print("No batch results to plot.")


 Processing PIDILITIND.NS ... 

/tmp/ipykernel_5655/3284724100.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True



MAPE=2.50%  R²=0.9389
  Forecasting next 5 trading days for PIDILITIND.NS ...
  Last known close: ₹1341.30  (date: 2026-03-30)
  Future Forecast:
    2026-03-31  →  ₹1329.42
    2026-04-01  →  ₹1332.39
    2026-04-02  →  ₹1333.74
    2026-04-03  →  ₹1329.13
    2026-04-04  →  ₹1327.67

 Processing HAVELLS.NS ... 

/tmp/ipykernel_5655/3284724100.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True



MAPE=5.24%  R²=0.8799
  Forecasting next 5 trading days for HAVELLS.NS ...
  Last known close: ₹1281.20  (date: 2026-03-30)
  Future Forecast:
    2026-03-31  →  ₹1350.84
    2026-04-01  →  ₹1358.78
    2026-04-02  →  ₹1361.10
    2026-04-03  →  ₹1358.01
    2026-04-04  →  ₹1358.18

 Processing DABUR.NS ... 

/tmp/ipykernel_5655/3284724100.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True



MAPE=3.34%  R²=0.7253
  Forecasting next 5 trading days for DABUR.NS ...
  Last known close: ₹430.70  (date: 2026-03-30)
  Future Forecast:
    2026-03-31  →  ₹452.14
    2026-04-01  →  ₹454.05
    2026-04-02  →  ₹454.69
    2026-04-03  →  ₹453.65
    2026-04-04  →  ₹453.56

Batch Zero-Shot Results
       Ticker  MAPE %     R²
PIDILITIND.NS    2.50 0.9389
   HAVELLS.NS    5.24 0.8799
     DABUR.NS    3.34 0.7253


## **MULTIPLE UNSEEN STOCK FUTURE FORECAST/PREDICTION**

In [16]:
print(f" Forecasting next {OUTPUT_LEN} trading days for stocks in BATCH_UNSEEN ...\n")
if 'mean_emb' not in globals() and model.use_embedding:
    mean_emb = model.embed.weight.mean(dim=0, keepdim=True).cpu().detach().clone()
elif not model.use_embedding:
    mean_emb = None

for current_ticker in BATCH_UNSEEN:
    print(f"\n--- Processing {current_ticker} ---")
    try:
        (
            X_curr_train, y_curr_train,
            X_curr_test,  y_curr_test,
            curr_dates
        ) = pipeline.prepare_single(current_ticker, fit_scaler=True)

        if len(X_curr_test) < 10:
            print(f"  Not enough data for {current_ticker}, skipping.")
            continue

        zs_preds_scaled = run_inference(model, X_curr_test, emb_override=mean_emb)
        zs_actual = inverse_close(y_curr_test[:, 0], current_ticker)
        zs_pred   = inverse_close(zs_preds_scaled[:, 0], current_ticker)

        zs_mape = np.mean(np.abs((zs_actual - zs_pred) / zs_actual)) * 100
        zs_r2   = r2_score(zs_actual, zs_pred)
        print(f"  Zero-Shot Metrics — {current_ticker}")
        print(f"    MAPE  : {zs_mape:.2f}%")
        print(f"    R²    : {zs_r2:.4f}")


        last_seq_curr = torch.tensor(X_curr_test[-1], dtype=torch.float32).unsqueeze(0).to(device)

        model.eval()
        with torch.no_grad():
            xp   = model.proj(last_seq_curr)
            xp   = model.pos(xp)

            if mean_emb is not None:
                xp   = xp + mean_emb.unsqueeze(1).to(device)
            xp   = model.trans(xp)
            fut_scaled_curr = model.fc(xp[:, -1, :]).cpu().numpy()[0]

        fut_prices_curr = inverse_close(fut_scaled_curr, current_ticker)
        last_date_curr    = curr_dates[-1]
        future_dates_curr = [last_date_curr + timedelta(days=i+1) for i in range(OUTPUT_LEN)]

        print(f"  Last known close: ₹{zs_actual[-1]:.2f}  (date: {last_date_curr.date()})")
        print("  Future Forecast:")
        for d, p in zip(future_dates_curr, fut_prices_curr):
            print(f"    {d.date()}  →  ₹{p:.2f}")

        N_plot_hist = 30

        plot_actual_dates = curr_dates[-N_plot_hist:]
        plot_actual_prices = zs_actual[-N_plot_hist:]
        plot_predicted_prices = zs_pred[-N_plot_hist:]

        forecast_start_date = curr_dates[-1]
        forecast_start_price = zs_pred[-1]

        forecast_dates_combined = [forecast_start_date] + future_dates_curr
        forecast_prices_combined = [forecast_start_price] + list(fut_prices_curr)

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=plot_actual_dates, y=plot_actual_prices,
            name="Actual Prices", mode="lines",
            line=dict(color="#00E5FF", width=2)
        ))

        fig.add_trace(go.Scatter(
            x=plot_actual_dates, y=plot_predicted_prices,
            name="Zero-Shot Predicted Prices", mode="lines",
            line=dict(color="#69FF47", width=2)
        ))

        fig.add_trace(go.Scatter(
            x=forecast_dates_combined,
            y=forecast_prices_combined,
            name=f"{OUTPUT_LEN}-Day Future Forecast", mode="lines",
            line=dict(color="#FFD700", width=2, dash='dot'),
            marker=dict(size=6, symbol='circle')
        ))

        fig.update_layout(
            title=f"{current_ticker} - Last {N_plot_hist} Days and {OUTPUT_LEN}-Day Forecast (Zero-Shot)",
            xaxis_title="Date",
            yaxis_title="Close Price (₹)",
            template="plotly_dark",
            legend=dict(x=1.02, y=1, xanchor='left', yanchor='top')
        )
        fig.show()

    except Exception as e:
        print(f"  Error processing {current_ticker}: {e}")

 Forecasting next 5 trading days for stocks in BATCH_UNSEEN ...


--- Processing PIDILITIND.NS ---


/tmp/ipykernel_5655/3284724100.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True



  Zero-Shot Metrics — PIDILITIND.NS
    MAPE  : 2.50%
    R²    : 0.9389
  Last known close: ₹1341.30  (date: 2026-03-30)
  Future Forecast:
    2026-03-31  →  ₹1329.42
    2026-04-01  →  ₹1332.39
    2026-04-02  →  ₹1333.74
    2026-04-03  →  ₹1329.13
    2026-04-04  →  ₹1327.67



--- Processing HAVELLS.NS ---


/tmp/ipykernel_5655/3284724100.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True



  Zero-Shot Metrics — HAVELLS.NS
    MAPE  : 5.24%
    R²    : 0.8799
  Last known close: ₹1281.20  (date: 2026-03-30)
  Future Forecast:
    2026-03-31  →  ₹1350.84
    2026-04-01  →  ₹1358.78
    2026-04-02  →  ₹1361.10
    2026-04-03  →  ₹1358.01
    2026-04-04  →  ₹1358.18



--- Processing DABUR.NS ---


/tmp/ipykernel_5655/3284724100.py:9: FutureWarning:

YF.download() has changed argument auto_adjust default to True



  Zero-Shot Metrics — DABUR.NS
    MAPE  : 3.34%
    R²    : 0.7253
  Last known close: ₹430.70  (date: 2026-03-30)
  Future Forecast:
    2026-03-31  →  ₹452.14
    2026-04-01  →  ₹454.05
    2026-04-02  →  ₹454.69
    2026-04-03  →  ₹453.65
    2026-04-04  →  ₹453.56
